# 在这个 notebook 中，你将学到：
1. 对你的 vLLM 服务运行真正的 GuideLLM 压测，并解读结果
2. 用 lm_eval 在标准任务上评估模型质量
3. 读懂已发布的 model card 数据，理解量化
4. 判断量化的权衡是否值得部署

## Setup

在这个 notebook 里，你会继续使用上一节课启动的同一个 vLLM 服务（模型 Qwen3-0.6B）。现在要回答：这个部署到底好不好？你要从两个维度测量：  
GuideLLM：性能维度问系统服务请求有多快、多高效  
lm_eval ：质量维度问模型在真实任务上表现多好

Let's send a quick test request to confirm everything is working.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import time, requests, json, os, glob
from openai import OpenAI

VLLM_URL = "http://localhost:8000"

for _ in range(12):
    try:
        r = requests.get(f"{VLLM_URL}/v1/models", timeout=5)
        if r.status_code == 200:
            MODEL = r.json()["data"][0]["id"]
            break
    except requests.ConnectionError:
        time.sleep(5)
else:
    raise RuntimeError("vLLM server not reachable.")

print(f"Connected to {VLLM_URL} — model: {MODEL}")

Connected to http://localhost:8000 — model: Qwen/Qwen3-0.6B


In [2]:
client = OpenAI(base_url=f"{VLLM_URL}/v1", api_key="unused")
resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", 
               "content": "What is model quantization in one sentence."}],
    max_tokens=30, temperature=0.7,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)
print(f"{MODEL}: {resp.choices[0].message.content.strip()}")

Qwen/Qwen3-0.6B: Model quantization is a technique used to reduce the size and computational complexity of neural networks by mapping high-dimensional weights to smaller, more efficient representations.


## Benchmarking with GuideLLM

[GuideLLM](https://github.com/neuralmagic/guidellm) 是 vLLM 项目的压测工具，用来评测推理性能。它按可控速率发请求，并记录每个请求的耗时。  

| Metric | What it measures |
|:--|:--|
| **TTFT** | 首token延迟：感知响应快慢 |
| **ITL** | ITL token间延迟：流式输出平滑度 |
| **E2E latency** | 端到端延迟：一个请求总时间 |
| **Throughput** | 每秒请求数  和 token数 |

你将跑同步压测（一次只发一个），共10个请求.

> **Note:** 10个是故意调得很小以便快速跑完，真实压测要用几百到几千个，或用--max-seconds按时间跑

In [3]:
os.makedirs("outputs", exist_ok=True)

In [ ]:
import subprocess

import sys
cmd = [
    sys.executable, "-m", "guidellm", "run",
    "--backend", f"kind=openai_http,target=http://localhost:8000,model={MODEL}",
    "--tokenizer", "kind=huggingface_auto,model=Qwen/Qwen3-0.6B",
    "--profile", "kind=synchronous",
    "--constraint", "kind=max_requests,count=10",
    "--data", "kind=synthetic_text,prompt_tokens=32,output_tokens=16",
    "--output", "kind=json,path=./outputs/benchmarks.json",
]

print(f"Running: {' '.join(cmd)}\n")

result = subprocess.run(cmd, capture_output=True, text=True, timeout=600)

if result.returncode == 0:
    print("Benchmark complete!")
    tail = result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout
    print(tail)
else:
    print(f"GuideLLM exited with code {result.returncode}")
    print(f"STDOUT:\n{result.stdout[-1000:]}")
    print(f"STDERR:\n{result.stderr[-1000:]}")

Running: /home/wty238388/ai-infra/vllm_env/bin/python -m guidellm run --backend kind=openai_http,target=http://localhost:8000,model=Qwen/Qwen3-0.6B --tokenizer kind=huggingface_auto,model=Qwen/Qwen3-0.6B --profile kind=synchronous --constraint kind=max_requests,count=10 --data kind=synthetic_text,prompt_tokens=32,output_tokens=16 --output kind=json,path=./outputs/benchmarks.json

Benchmark complete!
36m0.3     | 1.1    | 72.1 | 935.1 | 72.1 | 935.1 | 14.9 | 55.2 | 18.9 | 70.7 |
|=============|=========|========|======|=======|======|=======|======|======|======|======|


ℹ Server Throughput Statistics (All Requests)
|=============|=======|======|=========|==============|===============|==============|
| Benchmark   | Requests             ||| Input Tokens | Output Tokens | Total Tokens |
| Strategy    | Concurrency || Per Sec | Per Sec      | Per Sec       | Per Sec      |
|             | Mdn   | Mean | Mean                                               ||||
|-------------|-------|-----

## Interpreting Benchmark Results

 GuideLLM会把结果存成JSON，每个指标都预先算好统计值（均值、分位数、最小/最大），你直接读文件取数就行，不用自己算。

In [16]:
with open("outputs/benchmarks.json") as f:
    report = json.load(f)

bench = report["benchmarks"][0]
metrics = bench["metrics"]
n_requests = metrics["request_totals"]["successful"]

profile_type = bench.get("type") 
print(f"Profile: {profile_type}  |  Requests: {n_requests}\n")

display_metrics = {
    "TTFT (ms)":       "time_to_first_token_ms",
    "ITL (ms)":        "inter_token_latency_ms",
    "E2E Latency (s)": "request_latency",
    "Output tokens":   "output_token_count",
}

print(f"{'Metric':<20} {'Mean':>8} {'p50':>8} {'p95':>8} {'p99':>8}")
print("-" * 55)
for label, key in display_metrics.items():
    dist = metrics[key]["successful"]
    p = dist["percentiles"]
    print(f"{label:<20} {dist['mean']:>8.2f} {p['p50']:>8.2f} "
          f"{p['p95']:>8.2f} {p['p99']:>8.2f}")

throughput = metrics["output_tokens_per_second"]["successful"]
req_rate = metrics["requests_per_second"]["successful"]
print(f"\nThroughput: {req_rate['mean']:.2f} req/s  |  "
      f"{throughput['mean']:.1f} output tokens/s")

Profile: None  |  Requests: 10

Metric                   Mean      p50      p95      p99
-------------------------------------------------------
TTFT (ms)              162.02    72.13   935.11   935.11
ITL (ms)                18.46    14.95    55.22    55.22
E2E Latency (s)          0.44     0.30     1.13     1.13
Output tokens           16.00    16.00    16.00    16.00

Throughput: 2.27 req/s  |  46.2 output tokens/s


**Why Percentiles Matter**

均值会藏住离群点。平均100ms但p99是2s，意味着每100个用户就有1个等了20倍时间。  

| Percentile | Meaning |
|:--|:--|
| **p50 (median)** | 有一半请求比他快 |
| **p95** | 95% are faster — the remaining 5% are your "tail" |
| **p99** | 99% are faster — only 1 in 100 is slower |

评估部署永远看p95/p99，不只看均值。GuideLLM的指标自带分位数分解，对比mean和p95就能看出稳定性。.

## Evaluating Model Quality with lm_eval

性能压测只告诉你有多快，不告诉你答得好不好。.  

[lm_eval](https://github.com/EleutherAI/lm-evaluation-harness) 测任务表现：模型在标准学术集上答得多准.

| | GuideLLM | lm_eval |
|:--|:--|:--|
| **Measures** | Serving speed, latency, throughput | Task accuracy, quality |
| **Target** | A running inference server | A model (local or API)模型本身 |
| **Question** | "How well does this *deployment* perform?" | "How well does this *model* answer?模型准不准" |

你会用`local-completions`后端把lm_eval指向同一个vLLM服务——它走OpenAI兼容的completions接口，支持选择题必需的log-prob打分。你跑Hellaswag，它也在该模型量化论文里出现过，可直接对数。。.

> **Note:** 注意：用`simple_evaluate`只跑20条图快，生产要跑全量，甚至多遍.

In [17]:
import lm_eval

os.environ.setdefault("OPENAI_API_KEY", "unused")

TASK = "hellaswag"
print(f"Running lm_eval on {MODEL} via vLLM server ({TASK}, 20 examples)...\n")

results = lm_eval.simple_evaluate(
    model="local-completions",
    model_args=(
        f"model={MODEL},"
        f"base_url={VLLM_URL}/v1/completions,"
        "tokenized_requests=False,"
        "tokenizer=Qwen/Qwen3-0.6B,"
        "num_concurrent=1"
    ),
    tasks=[TASK],
    limit=20,
)

Running lm_eval on Qwen/Qwen3-0.6B via vLLM server (hellaswag, 20 examples)...



README.md:   0%|          | 0.00/7.02k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 24.4MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 6.11MB            

...

Generating test split:   0%|          | 0/10003 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/10042 [00:00<?, ? examples/s]

Map:   0%|          | 0/39905 [00:00<?, ? examples/s]

Map:   0%|          | 0/10042 [00:00<?, ? examples/s]

Requesting API: 100%|██████████| 80/80 [01:02<00:00,  1.29it/s]
fatal: not a git repository (or any of the parent directories): .git


In [18]:
task_results = results["results"][TASK]

print(f"Model: {MODEL}")
print(f"Task: {TASK}  |  Examples: 20\n")
for metric, value in task_results.items():
    if isinstance(value, (int, float)):
        print(f"  {metric}: {value:.4f}")

Model: Qwen/Qwen3-0.6B
Task: hellaswag  |  Examples: 20

  sample_len: 20.0000
  acc,none: 0.3000
  acc_stderr,none: 0.1051
  acc_norm,none: 0.3000
  acc_norm_stderr,none: 0.1051


## Reading Published Benchmark Data

In a previous lesson, you learned how to quantize a model with `llm-compressor` using GPTQ. In practice, quantized model publishers include **accuracy tables** on their model cards so users can evaluate the tradeoff without running every benchmark themselves.

Here's the accuracy table from the [RedHatAI/Qwen3-0.6B-quantized.w4a16 model card](https://huggingface.co/RedHatAI/Qwen3-0.6B-quantized.w4a16) — the W4A16 variant of the model you've been working with:

| Category | Benchmark | Qwen3-0.6B | W4A16 (this model) | Recovery |
|:--|:--|--:|--:|--:|
| **OpenLLM v1** | MMLU (5-shot) | 42.82 | 39.80 | 93.0% |
| | ARC Challenge (25-shot) | 32.85 | 30.72 | 93.5% |
| | GSM-8K (5-shot) | 1.82 | 2.20 | — |
| | Hellaswag (10-shot) | 43.04 | 41.02 | 95.3% |
| | Winogrande (5-shot) | 54.54 | 54.62 | 100.1% |
| | TruthfulQA (0-shot) | 51.61 | 48.77 | 94.5% |
| | **Average** | **37.78** | **36.19** | **95.8%** |
| **OpenLLM v2** | MMLU-Pro (5-shot) | 17.25 | 14.27 | — |
| | IFEval (0-shot) | 62.83 | 55.81 | 88.8% |
| | BBH (3-shot) | 4.23 | 1.63 | — |
| | Math-lvl-5 (4-shot) | 18.26 | 10.26 | — |
| | GPQA (0-shot) | 0.00 | 0.00 | — |
| | MuSR (0-shot) | 0.00 | 0.00 | — |
| | **Average** | **17.10** | **13.66** | — |
| **Multilingual** | MGSM (0-shot) | 19.70 | 19.90 | — |
| **Reasoning** | AIME 2024 | 9.69 | 3.44 | — |
| | AIME 2025 | 13.13 | 6.98 | — |
| | GPQA diamond | 29.29 | 27.78 | 94.8% |
| | Math-lvl-5 | 71.60 | 70.60 | 98.6% |
| | LiveCodeBench | 12.83 | 8.35 | — |

**Recovery**指量化版保留了基座几成精度。表中就是W4权4bit/A16激活16bit换体积/显存/吞吐，代价是精度掉一点。

## Making the Decision

You now have three sources of evidence:

| 来源                   | 它告诉你什么                               |
| -------------------- | ------------------------------------ |
| **GuideLLM**         | 你的部署**表现如何**：延迟、吞吐量、一致性等             |
| **lm_eval**          | 模型**回答得如何**：你自己实际运行的任务上的准确率          |
| **Model Card（模型卡片）** | 模型在**多个基准测试上的表现如何**：这些结果通常由模型发布者进行评测 |


当你决定是否部署一个量化模型时，需要同时考虑这两个维度。

一个优化方案如果能够让吞吐量翻倍，但准确率下降 15%，那么它可能并不值得。

反过来，一个非常准确的模型，如果无法满足延迟 SLO（服务等级目标），同样无法部署。

对于这个 W4A16 模型：

模型大小减少约 50%，同时在 OpenLLM v1 上的平均准确率损失约为 4%。

最终是否值得采用，取决于你的具体应用场景——你应该检查这个量化模型在对你真正重要的任务上的能力恢复情况。

## Summary


这个notebook你做了：  
1.用GuideLLM打vLLM测了TTFT、ITL、端到端和吞吐；  
2.用lm_eval跑Hellaswag测了质量，同样打运行中的服务；  
3.读了模型卡的精度表看清W4A16的完整 tradeoff；  
4.最后把速度证据和质量证据合起来做部署决策。

## Resources

- [GuideLLM GitHub](https://github.com/vllm-project/guidellm)
- [lm_eval](https://github.com/EleutherAI/lm-evaluation-harness)